In [3]:
import requests
import time
import json
import re
import random
from tqdm import tqdm
from google.colab import files

HEADERS = {
    "User-Agent": "MyColabBot/2.0 (inesgoddi@gmail.com)"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

API_SIMPLE = "https://simple.wikipedia.org/w/api.php"
API_EN = "https://en.wikipedia.org/w/api.php"

REQUEST_DELAY = 1.0
MAX_RETRIES = 8

DISCIPLINARY_ROOTS = [
    # Natural sciences
    "Physics", "Chemistry", "Biology", "Astronomy", "Earth science",

    # Formal sciences
    "Mathematics", "Computer science", "Statistics", "Logic",

    # Life sciences
    "Genetics", "Neuroscience", "Ecology", "Microbiology",
    "Zoology", "Botany",

    # Medical sciences
    "Medicine", "Surgery", "Pharmacology", "Epidemiology", "Public health",

    # Engineering
    "Engineering", "Mechanical engineering", "Electrical engineering",
    "Civil engineering", "Chemical engineering", "Aerospace engineering",

    # Social sciences
    "Economics", "Psychology", "Sociology", "Political science",
    "Anthropology", "Human geography",

    # Humanities
    "Philosophy", "History", "Literature", "Linguistics", "Theology",

    # Arts
    "Arts", "Music", "Visual arts", "Performing arts",
    "Film studies", "Architecture",

    # Interdisciplinary
    "Cognitive science", "Environmental science",
    "Data science", "Artificial intelligence", "Systems science"
]

BAD_CATEGORY_PATTERNS = [
    r"^Articles? ", r"^All articles?", r"^Wikipedia ", r"^Pages?",
    r"^CS1 ", r"^Harv ", r"^Use ", r"^Redirects?",
    r"^Disambiguation", r".*errors?$", r".*cleanup.*",
    r".*needing.*", r".*lacking.*", r".*unsourced.*",
    r".*unreferenced.*", r".*broken.*", r".*template.*",
    r".*infobox.*", r".*Wikidata.*", r".*stub.*",
    r".*stubs.*", r".*citation.*", r".*coordinates.*",
    r".*authority control.*", r".*public domain.*",
    r".*incorporating.*", r".*semi-protected.*",
    r".*plot summary.*", r".*bare URLs.*",
    r".*subscription.*", r".*maintenance.*",
    r".*hidden categor.*"
]

BAD_EXACT_CATEGORIES = {
    "Contents", "Lists", "Outlines", "Reference",
    "Comparisons", "Concepts", "Entities",
    "Objects", "Data", "Information"
}

BAD_CATEGORY_REGEX = [re.compile(p, re.IGNORECASE) for p in BAD_CATEGORY_PATTERNS]

DIGIT_PATTERN = re.compile(r"\d")
NON_ALPHA_PATTERN = re.compile(r"[^a-zA-Z\- ]")

BAD_WORDS = {
    "abuse", "abusive", "violence", "violent", "rape", "murder",
    "death", "torture", "prostitution", "porn",
    "sexual", "sex", "cannibal", "cannibalism", "drug",
    "addiction", "gambling", "terrorism", "crime", "criminal",
    "childhood", "children"
}

PEOPLE_PATTERN = re.compile(
    r"\b(bankers?|lawyers?|politicians?|actors?|singers?|players?|"
    r"scientists?|philosophers?|historians?|mathematicians?|"
    r"writers?|authors?|artists?|athletes?|accountants?)\b",
    re.IGNORECASE
)

IST_PATTERN = re.compile(r"\b\w+ist(s)?\b", re.IGNORECASE)
NAME_PATTERN = re.compile(r"^[A-Z][a-z]+ [A-Z][a-z]+$")


def is_disciplinary_category(cat):
    cat = cat.strip()
    cat_lower = cat.lower()

    if cat in BAD_EXACT_CATEGORIES:
        return False

    if any(p.search(cat) for p in BAD_CATEGORY_REGEX):
        return False

    if DIGIT_PATTERN.search(cat):
        return False

    if NON_ALPHA_PATTERN.search(cat):
        return False

    if any(word in cat_lower for word in BAD_WORDS):
        return False

    if PEOPLE_PATTERN.search(cat):
        return False

    if IST_PATTERN.search(cat):
        return False

    if NAME_PATTERN.match(cat):
        return False

    if len(cat.split()) > 1:
        return False
    # strict single-word (letters only, no spaces, no hyphens)
    if not re.fullmatch(r"[A-Za-z]+", cat):
      return False

    return True


def safe_get(api_url, params, max_retries=MAX_RETRIES):
    for attempt in range(max_retries):
        try:
            response = SESSION.get(api_url, params=params, timeout=30)

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    wait = int(retry_after)
                else:
                    wait = min(90, 5 * (2 ** attempt))

                wait += random.uniform(0, 2)
                print(f"429 rate limit. Waiting {wait:.1f}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()

            time.sleep(REQUEST_DELAY + random.uniform(0, 0.5))
            return response.json()

        except requests.exceptions.RequestException as e:
            wait = min(90, 3 * (2 ** attempt)) + random.uniform(0, 2)
            print(f"Request failed: {e}. Waiting {wait:.1f}s...")
            time.sleep(wait)

    return {}


def get_subcategories(category, api_url, depth=5):
    seen = set()
    to_visit = [(category, 0)]
    results = set()

    while to_visit:
        current_cat, current_depth = to_visit.pop()

        if current_cat in seen or current_depth > depth:
            continue

        seen.add(current_cat)
        cmcontinue = None

        while True:
            params = {
                "action": "query",
                "format": "json",
                "list": "categorymembers",
                "cmtitle": f"Category:{current_cat}",
                "cmnamespace": 14,
                "cmlimit": 500
            }

            if cmcontinue:
                params["cmcontinue"] = cmcontinue

            data = safe_get(api_url, params)

            subcats = [
                item["title"].replace("Category:", "")
                for item in data.get("query", {}).get("categorymembers", [])
            ]

            for subcat in subcats:
                if is_disciplinary_category(subcat):
                    results.add(subcat)

                    if current_depth < depth:
                        to_visit.append((subcat, current_depth + 1))

            cmcontinue = data.get("continue", {}).get("cmcontinue")

            if not cmcontinue:
                break

    return results


def fetch_root_subcats(cat):
    print(f"\nProcessing root category: {cat}")

    simple = get_subcategories(cat, API_SIMPLE, depth=7)
    en = get_subcategories(cat, API_EN, depth=7)

    return simple, en


print("Using predefined disciplinary root categories...")

subcategories_simple = set()
subcategories_en = set()

for cat in tqdm(DISCIPLINARY_ROOTS, desc="Getting subcategories"):
    simple, en = fetch_root_subcats(cat)
    subcategories_simple.update(simple)
    subcategories_en.update(en)

common_cats = sorted(
    (subcategories_simple & subcategories_en) | set(DISCIPLINARY_ROOTS)
)

common_cats = [
    cat for cat in common_cats
    if is_disciplinary_category(cat)
]

print(f"\nKept {len(common_cats)} disciplinary categories.")

with open("disciplinary_categories3.json", "w", encoding="utf-8") as f:
    json.dump(common_cats, f, ensure_ascii=False, indent=2)

files.download("disciplinary_categories3.json")

Using predefined disciplinary root categories...


Getting subcategories:   0%|          | 0/48 [00:00<?, ?it/s]


Processing root category: Physics


Getting subcategories:   2%|▏         | 1/48 [02:18<1:48:15, 138.21s/it]


Processing root category: Chemistry


...


Processing root category: Systems science


Getting subcategories: 100%|██████████| 48/48 [1:57:14<00:00, 146.55s/it]


Kept 242 disciplinary categories.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

with open("disciplinary_categories3.json", "r") as file:
    common_cats= json.load(file)

print(len(common_cats))

267


In [5]:

n = [
    "Anniversaries",
    "Architects",
    "Astronauts",
    "Astronomers",
    "Beekeepers",
    "Birthdays",
    "Choreographers",
    "Dancers",
    "Engineers",
    "Firefighters",
    "Islamophobia",
    "Plumbers",
    "Statisticians",
    "Surgeons",
    "Woodworkers",
    "Zoos"
]


result = [x for x in common_cats if x not in n]

print(len(result))

227


In [8]:
for i in result :
  print(i)

Acceleration
Adhesives
Algae
Amphibians
Angels
Animals
Annelids
Anthropology
Antisemitism
Aquariums
Archaeology
Architecture
Archives
Archosaurs
Arthropods
Arts
Astronomy
Autism
Bacteria
Beekeeping
Biology
Biomes
Birds
Birth
Books
Botany
Botnets
Brachiopods
Caecilians
Calendars
Carpenters
Cats
Centuries
Charts
Chemistry
Chromosomes
Chronology
Clocks
Cloning
Cockatoos
Cognition
Colors
Comics
Constellations
Construction
Costumes
Creativity
Cryptozoology
Dance
Dances
Days
Decades
Demolition
Demons
Deserts
Diagrams
Diapsids
Disasters
Documents
Dogs
Drama
Echinoderms
Ecology
Economics
Ecoregions
Ecosystems
Emotions
Engineering
Entomology
Epidemiology
Establishments
Ethnocentrism
Ethology
Eukaryotes
Extinction
Extremophiles
Fertilizers
Fire
Firefighting
Fires
Flags
Flatworms
Forestry
Forests
Frogs
Fungi
Gardening
Gardens
Gemstones
Genetics
Giftedness
Glaciers
Glass
Goods
Graphics
Grasslands
Happiness
Herbicides
Hippopotamuses
Historiography
History
Hydropower
Illusions
Infrastructure
Innovat

In [12]:
from numpy import char
import requests
import time
import json
import random
import csv
import re
from tqdm import tqdm
import pandas as pd
from google.colab import files

HEADERS = {
    "User-Agent": "MyColabBot/1.0 (inesgoddi@gmail.com)"
}

API_SIMPLE = "https://simple.wikipedia.org/w/api.php"
API_EN = "https://en.wikipedia.org/w/api.php"

# =========================
# 2. COLLECT ARTICLE TITLES
# =========================

matched_pairs = []

MAX_TOTAL_PAIRS = 10000
MAX_ARTICLES_PER_CATEGORY = 500

for cat in tqdm(result, desc="Collecting articles"):
    if len(cat.split()) >1 or any(char.isdigit() for char in cat):
        continue

    print(f"Collecting articles for category: {cat}")
    titles_simple = get_category_articles(
        cat,
        API_SIMPLE,
        limit=MAX_ARTICLES_PER_CATEGORY
    )

    if not titles_simple:
        continue

    existing_in_en = filter_existing_titles_in_en(titles_simple)

    count_for_cat = 0

    for title in existing_in_en:
        matched_pairs.append({
            "category": cat,
            "title": title
        })

        count_for_cat += 1

        if count_for_cat >= MAX_ARTICLES_PER_CATEGORY:
            break

        if len(matched_pairs) >= MAX_TOTAL_PAIRS:
            break

    if len(matched_pairs) >= MAX_TOTAL_PAIRS:
        break

with open("category_title_pairs_disciplinary3.json", "w", encoding="utf-8") as f:
    json.dump(matched_pairs, f, ensure_ascii=False, indent=2)

print(f"Collected {len(matched_pairs)} category-title pairs.")
print(matched_pairs)

Collected 6073 category-title pairs.
[{'category': 'Acceleration', 'title': 'Acceleration'}, {'category': 'Acceleration', 'title': 'Accelerometer'}, {'category': 'Acceleration', 'title': 'Centrifugal force'}, {'category': 'Acceleration', 'title': 'Centripetal force'}, {'category': 'Acceleration', 'title': 'Gravity'}, {'category': 'Acceleration', 'title': 'G-force'}, {'category': 'Adhesives', 'title': 'Adhesion'}, {'category': 'Adhesives', 'title': 'Adhesive'}, {'category': 'Adhesives', 'title': 'Binder (material)'}, {'category': 'Adhesives', 'title': 'Epoxy'}, {'category': 'Adhesives', 'title': 'Sticker'}, {'category': 'Adhesives', 'title': 'Cyanoacrylate'}, {'category': 'Adhesives', 'title': 'Thickening agent'}, {'category': 'Algae', 'title': 'Algae'}, {'category': 'Algae', 'title': 'Algal bloom'}, {'category': 'Algae', 'title': 'Brown algae'}, {'category': 'Algae', 'title': 'Charophyceae'}, {'category': 'Algae', 'title': 'Chlorella'}, {'category': 'Algae', 'title': 'Chlorophyta'}, {'

In [17]:
matched_pairs

[{'category': 'Acceleration', 'title': 'Acceleration'},
 {'category': 'Acceleration', 'title': 'Accelerometer'},
 {'category': 'Acceleration', 'title': 'Centrifugal force'},
 {'category': 'Acceleration', 'title': 'Centripetal force'},
 {'category': 'Acceleration', 'title': 'Gravity'},
 {'category': 'Acceleration', 'title': 'G-force'},
 {'category': 'Adhesives', 'title': 'Adhesion'},
 {'category': 'Adhesives', 'title': 'Adhesive'},
 {'category': 'Adhesives', 'title': 'Binder (material)'},
 {'category': 'Adhesives', 'title': 'Epoxy'},
 {'category': 'Adhesives', 'title': 'Sticker'},
 {'category': 'Adhesives', 'title': 'Cyanoacrylate'},
 {'category': 'Adhesives', 'title': 'Thickening agent'},
 {'category': 'Algae', 'title': 'Algae'},
 {'category': 'Algae', 'title': 'Algal bloom'},
 {'category': 'Algae', 'title': 'Brown algae'},
 {'category': 'Algae', 'title': 'Charophyceae'},
 {'category': 'Algae', 'title': 'Chlorella'},
 {'category': 'Algae', 'title': 'Chlorophyta'},
 {'category': 'Algae'

In [21]:
# =========================
# 3. EXTRACT SIMPLE + EN TEXTS
# =========================

results = []

for pair in tqdm(matched_pairs, desc="Extracting texts"):
    cat = pair["category"]
    title = pair["title"]

    simple_text = get_page_extract(title, API_SIMPLE)
    en_text = get_page_extract(title, API_EN)


    results.append({
        "category": cat,
        "title": title,
        "simple_text": simple_text,
        "en_text": en_text
    })

    time.sleep(0.1)

with open("category_title_texts_disciplinary3.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Extracted valid text pairs for {len(results)} articles.")

# =========================
# 4. CREATE PAIRWISE CSV
# =========================

output_rows = []

for entry in results:
    if entry is None or entry.get("simple_text") is None or entry.get("en_text") is None:
      continue
    simple_text = entry["simple_text"].strip()
    en_text = entry["en_text"].strip()
    category = entry["category"].strip()
    title = entry["title"].strip()

    label = random.randint(0, 1)

    if label == 1:
        sentence_1 = en_text
        sentence_2 = simple_text
    else:
        sentence_1 = simple_text
        sentence_2 = en_text

    output_rows.append({
        "sentence_1": sentence_1,
        "sentence_2": sentence_2,
        "label": label,
        "category": category,
        "title": title
    })

# Balance labels
label_0 = [row for row in output_rows if row["label"] == 0]
label_1 = [row for row in output_rows if row["label"] == 1]

min_len = min(len(label_0), len(label_1))

balanced_rows = label_0[:min_len] + label_1[:min_len]
random.shuffle(balanced_rows)

with open("output_with_disciplinary_category.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = [
        "sentence_1",
        "sentence_2",
        "label",
        "category",
        "title"
    ]

    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(balanced_rows)

print(f"Final balanced dataset size: {len(balanced_rows)}")
print("Saved: output_with_disciplinary_category.csv")

Extracting texts: 100%|██████████| 6073/6073 [5:39:23<00:00,  3.35s/it]


Extracted valid text pairs for 6073 articles.
Final balanced dataset size: 5460
Saved: output_with_disciplinary_category.csv


In [ ]:
count = 0
for i, entry in enumerate(results):
    if entry is None or entry.get("simple_text") is None:
        count += 1
print(count)

483


In [22]:
import pandas as pd

df = pd.read_csv("output_with_disciplinary_category.csv")
print(len(df))
print(df.head())

5460
                                          sentence_1  \
0  Decomposition is the process by which dead org...   
1  Chlorophyta or  chlorophytes is a major divisi...   
2  In logic and math, contraposition is the right...   
3  This is a list of food days by country. Many c...   
4  Photons  (from Greek φως, meaning light), in m...   

                                          sentence_2  label     category  \
0  Decomposition, or rotting, is what happens to ...      1      Biology   
1  Chlorophyta are a division of green algae.\nIt...      1        Algae   
2  In logic and mathematics, contraposition, or t...      0        Logic   
3  This is a list of food days by country. Many c...      1  Observances   
4  A photon (from Ancient Greek  φῶς, φωτός (phôs...      0        Light   

               title  
0      Decomposition  
1        Chlorophyta  
2     Contraposition  
3  List of food days  
4             Photon  


In [ ]:
df

,sentence_1,sentence_2,label,category,title
0,Archosauromorpha is a clade of diapsid reptile...,"Archosauromorpha (Greek for ""ruling lizard for...",0,Reptiles,Archosauromorpha
1,"Detective Conan (名探偵コナン, Meitantei Konan), als...","Case Closed, also officially known as Detectiv...",0,Manga,Case Closed
2,Consonant mutation is change in a consonant in...,Consonant mutation is a feature in languages w...,1,Linguistics,Consonant mutation
3,Xenophobia (from Ancient Greek ξένος (xénos) ...,Xenophobia is the fear or dislike of strangers...,1,Xenophobia,Xenophobia
4,"The meat ant (Iridomyrmex purpureus), also cal...","The meat ant (Iridomyrmex purpureus), also kno...",0,Ants,Meat ant
...,...,...,...,...,...
5803,The mole (symbol: mol) is the SI unit used to ...,The mole (symbol mol) is a unit of measurement...,0,Chemistry,Mole (unit)
5804,A siphon is a long tube-like structure that is...,A siphon is an anatomical structure which is p...,0,Molluscs,Siphon (mollusc)
5805,Shogun Warriors may refer to:\n\nShogun Warrio...,Shogun Warriors was a line of robot toyline re...,1,Robots,Shogun Warriors
5806,Discovery is the act of detecting something ne...,Discovery is the act of detecting something ne...,1,Cognition,Discovery (observation)


In [ ]:
pip install pylatexenc


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=865b298ce97e0d58254c26998a24cab787e6f6ab724473c354f59b4049bc73de
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [35]:
# ---------------------------
# Install required packages if needed
# ---------------------------
import subprocess
import sys


def install_and_import(package, import_name=None):
    """Install a missing package, then import it."""
    import_name = import_name or package

    try:
        return __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package]
        )
        return __import__(import_name)


ftfy = install_and_import("ftfy")
install_and_import("pylatexenc")
install_and_import("beautifulsoup4", "bs4")
install_and_import("tqdm")


# ---------------------------
# Imports
# ---------------------------
import csv
import html
import re
import unicodedata

import pandas as pd
from bs4 import BeautifulSoup
from pylatexenc.latex2text import LatexNodes2Text
from tqdm.auto import tqdm


tqdm.pandas()


# ---------------------------
# Optional: Mount Google Drive
# ---------------------------
"""
from google.colab import drive

drive.mount("/content/drive")

file_path = (
    "/content/drive/MyDrive/"
    "output_with_disciplinary_category.csv"
)
"""


# ---------------------------
# 1️⃣ Load CSV safely
# ---------------------------
file_path = "output_with_disciplinary_category.csv"

try:
    df1 = pd.read_csv(file_path, encoding="utf-8")
except UnicodeDecodeError:
    df1 = pd.read_csv(file_path, encoding="latin1")

print("CSV loaded. Rows:", len(df1))


# ---------------------------
# 2️⃣ Verify required columns
# ---------------------------
required_columns = {
    "sentence_1",
    "sentence_2",
    "label",
    "category",
    "title",
}

missing_columns = required_columns.difference(df1.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )


# ---------------------------
# 3️⃣ Preserve original content
# ---------------------------
df1["sentence_1_orig"] = df1["sentence_1"]
df1["sentence_2_orig"] = df1["sentence_2"]


# ---------------------------
# 4️⃣ Wikipedia section handling
# ---------------------------
TERMINAL_WIKIPEDIA_SECTIONS = [
    "See also",
    "Related pages",
    "Related articles",
    "References",
    "Reference",
    "Notes",
    "Footnotes",
    "Citations",
    "Sources",
    "Notes and references",
    "References and notes",
    "Notes and sources",
    "Notes and citations",
    "Bibliography",
    "Bibliographies",
    "Works cited",
    "Literature cited",
    "Further reading",
    "Further readings",
    "Suggested reading",
    "External links",
    "External link",
    "Other websites",
    "Other website",
]


WIKIPEDIA_HEADER_PATTERN = re.compile(
    r"""
    ^[ \t]*
    ={2,6}
    [ \t]*
    (.*?)
    [ \t]*
    ={2,6}
    [ \t]*
    $
    """,
    flags=re.IGNORECASE | re.MULTILINE | re.VERBOSE,
)


TERMINAL_SECTION_NAMES = {
    section.casefold()
    for section in TERMINAL_WIKIPEDIA_SECTIONS
}


def normalize_heading_name(heading):
    heading = re.sub(r"\s+", " ", heading)
    return heading.strip().casefold()


def remove_terminal_wikipedia_sections(text):
    """
    Remove the earliest terminal heading and everything after it.
    """
    if not text:
        return text, False

    for match in WIKIPEDIA_HEADER_PATTERN.finditer(text):
        heading_name = normalize_heading_name(match.group(1))

        if heading_name in TERMINAL_SECTION_NAMES:
            return text[:match.start()].rstrip(), True

    return text, False


def remove_nonterminal_wikipedia_headers(text):
    """
    Remove remaining heading lines while keeping their section content.
    """
    if not text:
        return text

    return WIKIPEDIA_HEADER_PATTERN.sub("", text)


# ---------------------------
# 5️⃣ Formula extraction and conversion
# ---------------------------
latex_converter = LatexNodes2Text(strict_latex=False)

# Formula forms commonly found in English and Simple English Wikipedia.
FORMULA_PATTERNS = [
    re.compile(r"<math\b[^>]*>.*?</math>", re.IGNORECASE | re.DOTALL),
    re.compile(r"\$\$.*?\$\$", re.DOTALL),
    re.compile(r"\\\[.*?\\\]", re.DOTALL),
    re.compile(r"\\\(.*?\\\)", re.DOTALL),
    re.compile(r"(?<!\$)\$(?!\$).*?(?<!\$)\$(?!\$)", re.DOTALL),
    re.compile(
        r"\{\{\s*(?:math|mvar|nowrap)\s*\|.*?\}\}",
        re.IGNORECASE | re.DOTALL,
    ),
]

FORMULA_TOKEN_PATTERN = re.compile(r"⟪FORMULA_(\d+)⟫")

# This detects LaTeX-like material that survived conversion.
LEFTOVER_FORMULA_PATTERN = re.compile(
    r"""
    \\(?:begin|end|frac|sqrt|text|mathrm|mathbf|mathit|
       operatorname|displaystyle|ce|left|right)\b
    |
    \\[A-Za-z]+
    |
    \$\$.*?\$\$
    |
    (?<!\$)\$(?!\$).*?(?<!\$)\$(?!\$)
    |
    \\\[.*?\\\]
    |
    \\\(.*?\\\)
    |
    <math\b
    |
    </?m(?:ath|row|i|n|o|frac|sup|sub|sqrt|text)\b
    """,
    flags=re.IGNORECASE | re.DOTALL | re.VERBOSE,
)


def extract_formulas(text):
    """
    Replace formulas with stable tokens before HTML cleanup.

    This prevents BeautifulSoup or other cleanup steps from deleting them.
    """
    formulas = []

    def save_formula(match):
        token = f"⟪FORMULA_{len(formulas)}⟫"
        formulas.append(match.group(0))
        return token

    for pattern in FORMULA_PATTERNS:
        text = pattern.sub(save_formula, text)

    return text, formulas


def convert_mathml_to_text(markup):
    """
    Convert common MathML structures into readable plain-text notation.
    """
    soup = BeautifulSoup(markup, "html.parser")

    for fraction in soup.find_all("mfrac"):
        parts = fraction.find_all(recursive=False)

        if len(parts) >= 2:
            numerator = parts[0].get_text(" ", strip=True)
            denominator = parts[1].get_text(" ", strip=True)
            fraction.replace_with(f"({numerator})/({denominator})")

    for tag_name, operator in (("msup", "^"), ("msub", "_")):
        for tag in soup.find_all(tag_name):
            parts = tag.find_all(recursive=False)

            if len(parts) >= 2:
                base = parts[0].get_text(" ", strip=True)
                value = parts[1].get_text(" ", strip=True)
                tag.replace_with(f"{base}{operator}({value})")

    for root in soup.find_all("msqrt"):
        root.replace_with(
            f"√({root.get_text(' ', strip=True)})"
        )

    return soup.get_text(" ", strip=True)


def fallback_latex_to_text(text):
    """
    Convert common LaTeX commands that may remain after pylatexenc.
    """
    replacements = {
        r"\\times\b": "×",
        r"\\cdot\b": "·",
        r"\\pm\b": "±",
        r"\\leq?\b": "≤",
        r"\\geq?\b": "≥",
        r"\\neq\b": "≠",
        r"\\approx\b": "≈",
        r"\\rightarrow\b": "→",
        r"\\to\b": "→",
        r"\\leftarrow\b": "←",
        r"\\leftrightarrow\b": "↔",
        r"\\infty\b": "∞",
        r"\\alpha\b": "α",
        r"\\beta\b": "β",
        r"\\gamma\b": "γ",
        r"\\delta\b": "δ",
        r"\\epsilon\b": "ε",
        r"\\theta\b": "θ",
        r"\\lambda\b": "λ",
        r"\\mu\b": "μ",
        r"\\pi\b": "π",
        r"\\rho\b": "ρ",
        r"\\sigma\b": "σ",
        r"\\phi\b": "φ",
        r"\\omega\b": "ω",
        r"\\sum\b": "Σ",
        r"\\prod\b": "Π",
        r"\\int\b": "∫",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)

    previous = None

    while previous != text:
        previous = text

        text = re.sub(
            r"\\(?:text|textrm|mathrm|mathbf|mathit|operatorname)"
            r"\s*\{([^{}]*)\}",
            r"\1",
            text,
        )

        text = re.sub(
            r"\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}",
            r"(\1)/(\2)",
            text,
        )

        text = re.sub(
            r"\\sqrt\s*\{([^{}]*)\}",
            r"√(\1)",
            text,
        )

        text = re.sub(
            r"([A-Za-z0-9)\]])\s*\^\s*\{([^{}]+)\}",
            r"\1^(\2)",
            text,
        )

        text = re.sub(
            r"([A-Za-z0-9)\]])\s*_\s*\{([^{}]+)\}",
            r"\1_(\2)",
            text,
        )

    text = text.replace(r"\left", "")
    text = text.replace(r"\right", "")
    text = text.replace(r"\,", " ")
    text = text.replace(r"\;", " ")
    text = text.replace(r"\:", " ")
    text = text.replace(r"\!", "")

    return text


def unwrap_formula(formula):
    """
    Remove formula wrappers while retaining the formula itself.
    """
    formula = formula.strip()

    if re.match(r"<math\b", formula, flags=re.IGNORECASE):
        inner = re.sub(
            r"^\s*<math\b[^>]*>|</math>\s*$",
            "",
            formula,
            flags=re.IGNORECASE | re.DOTALL,
        )

        if re.search(
            r"</?m(?:ath|row|i|n|o|frac|sup|sub|sqrt|text)\b",
            inner,
            flags=re.IGNORECASE,
        ):
            return convert_mathml_to_text(inner)

        return inner

    if formula.startswith("$$") and formula.endswith("$$"):
        return formula[2:-2]

    if formula.startswith(r"\[") and formula.endswith(r"\]"):
        return formula[2:-2]

    if formula.startswith(r"\(") and formula.endswith(r"\)"):
        return formula[2:-2]

    if formula.startswith("$") and formula.endswith("$"):
        return formula[1:-1]

    if formula.startswith("{{") and formula.endswith("}}"):
        match = re.match(
            r"^\{\{\s*[^|{}]+\|\s*(.*?)\s*\}\}$",
            formula,
            flags=re.DOTALL,
        )

        if match:
            return match.group(1)

    return formula


def convert_formula(formula):
    """
    Convert one formula to readable plain text.

    The original formula is returned if conversion cannot be completed,
    so formulas are never silently deleted.
    """
    original_formula = formula

    try:
        formula = unwrap_formula(formula)

        # Preserve the contents of MediaWiki math wrappers.
        formula = re.sub(
            r"\{\\displaystyle\s*(.*?)\}",
            r"\1",
            formula,
            flags=re.DOTALL,
        )

        formula = re.sub(
            r"\{?\\ce\s*\{(.*?)\}\}?",
            r"\1",
            formula,
            flags=re.DOTALL,
        )

        converted = latex_converter.latex_to_text(formula)
        converted = fallback_latex_to_text(converted)

        # A second pass often resolves commands exposed by the fallback.
        converted = latex_converter.latex_to_text(converted)
        converted = html.unescape(converted)
        converted = unicodedata.normalize("NFKC", converted)
        converted = re.sub(r"[ \t]+", " ", converted).strip()

        if not converted:
            return original_formula, "conversion produced empty text"

        leftover = LEFTOVER_FORMULA_PATTERN.search(converted)

        if leftover:
            return converted, (
                "formula markup remains: "
                f"{leftover.group(0).replace(chr(10), ' ')[:120]}"
            )

        return converted, ""

    except Exception as exc:
        return original_formula, (
            f"{type(exc).__name__}: {exc}"
        )


def restore_formulas(text, formulas):
    """
    Restore all formula tokens and return conversion diagnostics.
    """
    errors = []

    def restore(match):
        formula_index = int(match.group(1))

        if formula_index >= len(formulas):
            errors.append(
                f"missing formula for token {formula_index}"
            )
            return match.group(0)

        converted, error = convert_formula(
            formulas[formula_index]
        )

        if error:
            errors.append(
                f"formula {formula_index}: {error}"
            )

        return converted

    restored = FORMULA_TOKEN_PATTERN.sub(restore, text)

    remaining_tokens = FORMULA_TOKEN_PATTERN.findall(restored)

    if remaining_tokens:
        errors.append(
            f"unrestored formula tokens: {remaining_tokens}"
        )

    return restored, "; ".join(errors)


# ---------------------------
# 6️⃣ Cleaning function with diagnostics
# ---------------------------
def clean_text_with_reason(text):
    if pd.isna(text):
        return "", "original_nan", False, ""

    original = str(text)
    cleaned = ftfy.fix_text(original)

    # Protect formulas before parsing HTML or removing markup.
    cleaned, formulas = extract_formulas(cleaned)

    # Convert HTML into visible plain text.
    cleaned = BeautifulSoup(
        cleaned,
        "html.parser",
    ).get_text(separator="\n")

    # Remove the earliest terminal section and everything after it.
    cleaned, terminal_section_removed = (
        remove_terminal_wikipedia_sections(cleaned)
    )

    # Remove non-terminal heading lines only.
    cleaned = remove_nonterminal_wikipedia_headers(cleaned)

    # Restore and convert every extracted formula.
    cleaned, formula_error = restore_formulas(
        cleaned,
        formulas,
    )

    # Try converting any unwrapped LaTeX still present in the prose.
    try:
        cleaned = latex_converter.latex_to_text(cleaned)
        cleaned = fallback_latex_to_text(cleaned)
    except Exception as exc:
        extra_error = (
            f"whole-text conversion: {type(exc).__name__}: {exc}"
        )
        formula_error = (
            f"{formula_error}; {extra_error}"
            if formula_error
            else extra_error
        )

    cleaned = html.unescape(cleaned)
    cleaned = unicodedata.normalize("NFKC", cleaned)

    for character in [
        "\u200b",
        "\u200c",
        "\u200d",
        "\u2060",
        "\u2061",
        "\ufeff",
    ]:
        cleaned = cleaned.replace(character, "")

    cleaned = re.sub(r"[ \t]+", " ", cleaned)
    cleaned = re.sub(r" *\n *", "\n", cleaned)
    cleaned = re.sub(r"\n{2,}", "\n", cleaned)
    cleaned = cleaned.strip()

    leftover_match = LEFTOVER_FORMULA_PATTERN.search(cleaned)

    if leftover_match:
        leftover_error = (
            "unconverted formula markup: "
            f"{leftover_match.group(0).replace(chr(10), ' ')[:120]}"
        )
        formula_error = (
            f"{formula_error}; {leftover_error}"
            if formula_error
            else leftover_error
        )

    formula_failed = bool(formula_error)

    if original.strip() and not cleaned:
        reason = "content_removed_by_cleaning"
    elif formula_failed:
        reason = "formula_conversion_warning"
    elif terminal_section_removed:
        reason = "terminal_section_removed"
    else:
        reason = ""

    return cleaned, reason, formula_failed, formula_error


# ---------------------------
# 7️⃣ Apply cleaning with progress bars
# ---------------------------
def clean_column(column_name):
    tqdm.pandas(
        desc=f"Cleaning {column_name}",
        unit="row",
    )

    results = df1[column_name].progress_apply(
        clean_text_with_reason
    )

    return pd.DataFrame(
        results.tolist(),
        index=df1.index,
        columns=[
            f"{column_name}_clean",
            f"{column_name}_reason",
            f"{column_name}_formula_failed",
            f"{column_name}_formula_error",
        ],
    )


sentence_1_results = clean_column("sentence_1")
sentence_2_results = clean_column("sentence_2")

df1 = pd.concat(
    [
        df1,
        sentence_1_results,
        sentence_2_results,
    ],
    axis=1,
)


# ---------------------------
# 8️⃣ Convert destroyed content to NaN
# ---------------------------
for column in ["sentence_1", "sentence_2"]:
    clean_column_name = f"{column}_clean"
    reason_column = f"{column}_reason"

    destroyed_mask = (
        df1[reason_column] == "content_removed_by_cleaning"
    )

    df1.loc[destroyed_mask, clean_column_name] = pd.NA


# ---------------------------
# 9️⃣ Save formula warnings for inspection
# ---------------------------
formula_warning_mask = (
    df1["sentence_1_formula_failed"]
    | df1["sentence_2_formula_failed"]
)

formula_warnings = df1.loc[
    formula_warning_mask,
    [
        "title",
        "sentence_1_orig",
        "sentence_1_clean",
        "sentence_1_formula_error",
        "sentence_2_orig",
        "sentence_2_clean",
        "sentence_2_formula_error",
    ],
].copy()

formula_warning_output_path = "formula_conversion_warnings.csv"

formula_warnings.to_csv(
    formula_warning_output_path,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
)

print(
    f"\nFormula warning report saved to: "
    f"{formula_warning_output_path} "
    f"({len(formula_warnings)} rows)"
)


# ---------------------------
# 🔟 Report cleaning results
# ---------------------------
for column in ["sentence_1", "sentence_2"]:
    reason_column = f"{column}_reason"
    formula_failed_column = f"{column}_formula_failed"

    destroyed_count = (
        df1[reason_column] == "content_removed_by_cleaning"
    ).sum()

    terminal_count = (
        df1[reason_column] == "terminal_section_removed"
    ).sum()

    formula_warning_count = (
        df1[formula_failed_column].sum()
    )

    print(f"\nResults for {column}:")
    print(f"  Content completely removed: {destroyed_count}")
    print(f"  Terminal sections removed: {terminal_count}")
    print(f"  Formula conversion warnings: {formula_warning_count}")


# ---------------------------
# 1️⃣1️⃣ Remove unusable rows
# ---------------------------
rows_before_drop = len(df1)

df1 = df1.dropna(
    subset=[
        "sentence_1_clean",
        "sentence_2_clean",
    ]
)

nonempty_mask = (
    df1["sentence_1_clean"].str.strip().ne("")
    &
    df1["sentence_2_clean"].str.strip().ne("")
)

df1 = (
    df1.loc[nonempty_mask]
    .copy()
    .reset_index(drop=True)
)

removed_rows = rows_before_drop - len(df1)

print(f"\nRemoved unusable rows: {removed_rows}")
print(f"Remaining rows: {len(df1)}")


# ---------------------------
# 1️⃣2️⃣ Replace original sentences
# ---------------------------
df1["sentence_1"] = df1["sentence_1_clean"]
df1["sentence_2"] = df1["sentence_2_clean"]

df1 = df1[
    [
        "sentence_1",
        "sentence_2",
        "label",
        "category",
        "title",
    ]
]


# ---------------------------
# 1️⃣3️⃣ Save cleaned dataframe
# ---------------------------
output_path = "output_cleaned_title.csv"

df1.to_csv(
    output_path,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_MINIMAL,
)

print(f"\nCleaned CSV saved to: {output_path}")

CSV loaded. Rows: 5460


Cleaning sentence_1:   0%|          | 0/5460 [00:00<?, ?row/s]

Cleaning sentence_2:   0%|          | 0/5460 [00:00<?, ?row/s]


Formula warning report saved to: formula_conversion_warnings.csv (3 rows)

Results for sentence_1:
  Content completely removed: 0
  Terminal sections removed: 4762
  Formula conversion warnings: 1

Results for sentence_2:
  Content completely removed: 0
  Terminal sections removed: 4749
  Formula conversion warnings: 2

Removed unusable rows: 0
Remaining rows: 5460

Cleaned CSV saved to: output_cleaned_title.csv


In [36]:
df1

,sentence_1,sentence_2,label,category,title
0,Decomposition is the process by which dead org...,"Decomposition, or rotting, is what happens to ...",1,Biology,Decomposition
1,Chlorophyta or chlorophytes is a major divisio...,Chlorophyta are a division of green algae.\nIt...,1,Algae,Chlorophyta
2,"In logic and math, contraposition is the right...","In logic and mathematics, contraposition, or t...",0,Logic,Contraposition
3,This is a list of food days by country. Many c...,This is a list of food days by country. Many c...,1,Observances,List of food days
4,"Photons (from Greek φως, meaning light), in ma...","A photon (from Ancient Greek φῶς, φωτός (phôs,...",0,Light,Photon
...,...,...,...,...,...
5455,Gapo is a Vietnamese social networking service...,Gapo is a social networking platform which is ...,1,Software,Gapo
5456,Horticulture is the practical botany of garden...,Horticulture (from Latin: horti + culture) is ...,0,Gardening,Horticulture
5457,Spaceflight is when an object (or spacecraft) ...,Spaceflight (also space flight) is an applicat...,0,Spaceflight,Spaceflight
5458,"A sexual fantasy is a mental image, or a patte...","A sexual fantasy, or erotic fantasy, is an aut...",0,Literature,Sexual fantasy


In [ ]:
from google.colab import files
files.download('output_cleaned_title.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
import pandas as pd

df = pd.read_csv("output_cleaned_title.csv")
print(len(df))
print(df.head())

5460
                                          sentence_1  \
0  Decomposition is the process by which dead org...   
1  Chlorophyta or chlorophytes is a major divisio...   
2  In logic and math, contraposition is the right...   
3  This is a list of food days by country. Many c...   
4  Photons (from Greek φως, meaning light), in ma...   

                                          sentence_2  label     category  \
0  Decomposition, or rotting, is what happens to ...      1      Biology   
1  Chlorophyta are a division of green algae.\nIt...      1        Algae   
2  In logic and mathematics, contraposition, or t...      0        Logic   
3  This is a list of food days by country. Many c...      1  Observances   
4  A photon (from Ancient Greek φῶς, φωτός (phôs,...      0        Light   

               title  
0      Decomposition  
1        Chlorophyta  
2     Contraposition  
3  List of food days  
4             Photon  


In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

with open('/content/drive/MyDrive/data/output_cleaned_title.csv', 'r') as f:
    df = pd.read_csv(f)



print("number of label 0 rows :", (df['label'] == 0).sum())
print("number of label 1 rows :", (df['label'] == 1).sum())

Mounted at /content/drive
number of label 0 rows : 2730
number of label 1 rows : 2730
